In [1]:
import numpy as np, pandas as pd, os, glob



In [2]:
sample_paths = glob.glob(os.path.join("**", "sample_submission.csv"), recursive=True)
if not sample_paths:
    raise FileNotFoundError("sample_submission.csv not found in any subdirectory.")
df = pd.read_csv(sample_paths[0])



In [3]:
train_paths = glob.glob(os.path.join("**", "train.json"), recursive=True)
if not train_paths:
    raise FileNotFoundError("train.json not found in any subdirectory.")
train_df = pd.read_json(train_paths[0], lines=True)

targets = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]

# overall weighted mean using all samples
weights_all = train_df["signal_to_noise"].values
overall_means = {}
for t in targets:
    stacked_all = np.vstack(train_df[t].values)
    overall_means[t] = np.average(
        stacked_all.ravel(), weights=np.repeat(weights_all, stacked_all.shape[1])
    )

# high‑quality subset for per‑position statistics
hq_df = train_df[train_df["SN_filter"] == 1]
if hq_df.empty:
    hq_df = train_df

# per‑position robust median (less sensitive to outliers)
per_pos_medians = {}
for t in targets:
    stacked_hq = np.vstack(hq_df[t].values)
    per_pos_medians[t] = np.median(stacked_hq, axis=0)



In [4]:
# reduced blending factor to rely more on stable overall means
alpha = 0.6


def get_pred_vals(pos):
    """Blended prediction for all targets at a given sequence position."""
    if pos < 68:
        return [
            alpha * per_pos_medians[t][pos] + (1 - alpha) * overall_means[t]
            for t in targets
        ]
    else:
        return [overall_means[t] for t in targets]


pred_positions = df["id_seqpos"].apply(lambda x: int(x.split("_")[-1]))

for i, t in enumerate(targets, start=1):
    df[t] = pred_positions.apply(lambda p: get_pred_vals(p)[i - 1])



In [5]:
df.to_csv("submission.csv", index=False)